# FieldVision on Colab

Runs the 2023 pipeline (`python -m src.run_all`) on a Colab machine, which has a faster link to Planetary Computer and gridMET than a laptop on campus Wi-Fi.

Before the first cell: **Runtime → Change runtime type → High-RAM**. A free session drops after a long idle stretch, and the disk is deleted when it disconnects. The last cell copies `data/processed` to Google Drive.

BARC Sentinel-2, Sentinel-1, and weather only need the GeoPackage already in the repo. County collect also needs the 2023 Cropland Data Layer and the national Crop Sequence Boundaries file. Those are several gigabytes and are not in git. Put them on Drive if you run the county cells.

Open this file in Colab with **File → Upload notebook**, or, after it is on GitHub, https://colab.research.google.com/github/ChenSiyun1234/FieldVision/blob/dataset-analysis/notebooks/colab_pipeline.ipynb

## 1. Get the code

The clone uses the `dataset-analysis` branch on GitHub. Local edits that are not pushed (the NDMI and band-median columns, and the load progress bar) are not in that clone. Push the branch first, or upload a zip of the project to `/content/FieldVision` and skip the clone.

Leave out `.venv`, `notebooks/dataset/Cropland Data Layer Releases 2023`, and `notebooks/dataset/NationalCSB_2016-2023_rev23`. Together those are about 14 GB.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/FieldVision")
if not (REPO / "src" / "run_all.py").exists():
    !git clone --branch dataset-analysis https://github.com/ChenSiyun1234/FieldVision.git /content/FieldVision
os.chdir(REPO)
print(Path.cwd())


## 2. Install

Colab already has some of these. Installing from `requirements.txt` matches the local environment.

In [ ]:
!pip install -q -r requirements.txt


## 3. Google Drive

Results are copied here at the end. County rasters, if you have them, go in the same folder:

- `MyDrive/FieldVision/raw/cdl/2023_30m_cdls.tif`
- `MyDrive/FieldVision/raw/csb/NationalCSB_2016-2023_rev23/` (the folder that contains `CSB1623.gdb`)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FieldVision")
DRIVE.mkdir(parents=True, exist_ok=True)

def link(src: Path, dest: Path) -> None:
    if dest.exists() or dest.is_symlink():
        print(f"already present: {dest}")
        return
    if not src.exists():
        print(f"not on Drive, skipping: {src}")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.symlink_to(src)
    print(f"linked {dest} -> {src}")

link(DRIVE / "raw/cdl/2023_30m_cdls.tif", REPO / "data/raw/cdl/2023_30m_cdls.tif")
link(
    DRIVE / "raw/csb/NationalCSB_2016-2023_rev23",
    REPO / "data/raw/csb/NationalCSB_2016-2023_rev23",
)


## 4. BARC

One tile, 19 labeled fields, May 1–Dec 15 2023. On a laptop the Sentinel-2 pass was about 13 minutes and Sentinel-1 about 4 minutes. Weather then downloads three national gridMET files (maximum temperature, minimum temperature, precipitation), about 150 MB each, and samples the field centroids.

Run the cells one at a time. Add `--test` to the command if you only want 4 fields.

In [ ]:
!python -u -m src.run_all --step s2 --region barc


In [ ]:
!python -u -m src.run_all --step s1 --region barc


In [ ]:
!python -u -m src.run_all --step weather --region barc


## 5. Transfer counties

Leave `RUN_COUNTIES` false unless the CDL and the national Crop Sequence Boundaries folder are linked above. `extract_csb` writes one GeoPackage per state and skips a file that already exists. Collect for Lancaster kept about 3,700 fields locally, so a full Sentinel-2 run for both counties is many hours. Use `--test` first.

In [ ]:
RUN_COUNTIES = False

if RUN_COUNTIES:
    !python -u -m src.extract_csb
    !python -u -m src.run_all --step collect --region all
    # Then, per county, for example:
    # !python -u -m src.run_all --step s2 --region pennsylvania_42071 --test
else:
    print("County steps skipped. Set RUN_COUNTIES = True after the rasters are on Drive.")


## 6. Keep the CSVs

Copies `data/processed` and `DATA_LOG.md` to `MyDrive/FieldVision`. Run this before you close the tab.

In [ ]:
import shutil

shutil.copytree(REPO / "data/processed", DRIVE / "processed", dirs_exist_ok=True)
shutil.copy2(REPO / "DATA_LOG.md", DRIVE / "DATA_LOG.md")
print("saved to", DRIVE)
